# Fully Homomorphic Caesar Cipher (FHE Caesar Cipher)

Implementing Fully Homomorphic computing using a caesar cipher.

The Caesar Cipher itself is completely insecure due to:
- frequency analysis (the distribution of values remains the same, it's just shifted)

However, if we instead used a One Time pad for the shifts, surely Shannon's proof gives us \
information-theoretic guarantees that it's secure?

## Addition

In [50]:
def encrypt(ll, shift, mod):
    return [(i+shift) % mod for i in ll]

def decrypt(ll, shift, mod):
    return [(i-shift) % mod for i in ll]

def add(a, b, mod):
    assert len(a) == len(b)
    return [(a[i] + b[i]) % mod for i in range(len(a))]

ll = [5000]
k = 100
mod = 10_000_000
a = encrypt(ll, k, mod)
b = encrypt(ll, k, mod)
s = add(a, b, mod)

a, b, s, decrypt(s, k*2, mod)

([5100], [5100], [10200], [10000])

## Multiplication

In [43]:
# 1x1 to 2x2
table = [
    [1, 1, 1],
    [1, 2, 2],
    [2, 1, 2],
    [2, 2, 4]
]

print(table)

# preshift table with key?
for i in range(len(table)): table[i] = encrypt(table[i], k, mod)

print("tb:", table)

def multiply(a, b, table):
    assert len(a) == len(b)
    def product(b, c, table):
        print(b, c, table)
        # assuming table has product
        return [row[2] for row in table if row[0] == b and row[1] == c][0]
    
    return sum([product(a[i], b[i], table) for i in range(len(a))])

multiply(a, b, table) - (k * len(a))

[[1, 1, 1], [1, 2, 2], [2, 1, 2], [2, 2, 4]]
tb: [[3, 3, 3], [3, 4, 4], [4, 3, 4], [4, 4, 6]]
4 4 [[3, 3, 3], [3, 4, 4], [4, 3, 4], [4, 4, 6]]


4

## One-Hot Lookup

In [44]:
import numpy as np

table = [
    [1, 1, 1],
    [1, 2, 2],
    [2, 1, 2],
    [2, 2, 4]
]

def gen_onehot_table(entries):
    table = []
    flatten = lambda ll: [i for x in ll for i in x]
    items = flatten(entries)
    items = set(items)
    l = len(items)
    items = list(items)
    items.sort()
    mapping = {idx:itm for itm, idx in enumerate(items)}
    print(mapping, items)

    def one_hot(v, l, d=[0, 1]):
        print(v)
        l = [d[0] if i != v-1 else d[1] for i in range(l)][::-1]
        return l
    
    for r in entries:
        a, b, c = r
        new_a = one_hot(mapping[a]+1, l)
        new_b = one_hot(mapping[b]+1, l)
        new_c = one_hot(mapping[c]+1, l)
        new_c[new_c.index(1)] = c
        table.append([new_a, new_b, new_c])

    return table

a, b = [0, 0, 1], [0, 0, 1]

mult_table = gen_onehot_table(table)

def match(x, y, tbl):
    for r in tbl:
        a, b, c = r
        i = np.dot(a, x)
        j = np.dot(b, y)
        print("ij:", a, b, c, i, j)

print(mult_table)

match(a, b, mult_table)

{1: 0, 2: 1, 4: 2} [1, 2, 4]
1
1
1
1
2
2
2
1
2
2
2
3
[[[0, 0, 1], [0, 0, 1], [0, 0, 1]], [[0, 0, 1], [0, 1, 0], [0, 2, 0]], [[0, 1, 0], [0, 0, 1], [0, 2, 0]], [[0, 1, 0], [0, 1, 0], [4, 0, 0]]]
ij: [0, 0, 1] [0, 0, 1] [0, 0, 1] 1 1
ij: [0, 0, 1] [0, 1, 0] [0, 2, 0] 1 0
ij: [0, 1, 0] [0, 0, 1] [0, 2, 0] 0 1
ij: [0, 1, 0] [0, 1, 0] [4, 0, 0] 0 0


## Multi One-Hot Lookup

In [45]:
import random

mod = 10_000_007  # prime modulus
domain = 4        # values: 0, 1, 2, 3

# multiplication table: T[i][j] = i * j
T = [[i * j for j in range(domain)] for i in range(domain)]

def one_hot(val, size):
    return [1 if i == val else 0 for i in range(size)]

def encrypt_one_hot(oh, mod):
    keys = [random.randint(0, mod - 1) for _ in oh]
    enc = [(v + k) % mod for v, k in zip(oh, keys)]
    return enc, keys

# === TEST ALL PAIRS ===
for a in range(domain):
    for b in range(domain):
        a_oh = one_hot(a, domain)
        b_oh = one_hot(b, domain)

        E_a, K_a = encrypt_one_hot(a_oh, mod)
        E_b, K_b = encrypt_one_hot(b_oh, mod)

        # --- SERVER (knows T, sees E_a and E_b, knows nothing else) ---

        # Step 1: select row using encrypted one-hot a
        row = [sum(T[i][j] * E_a[i] for i in range(domain)) % mod
               for j in range(domain)]

        # Step 2: use row values as constants against encrypted one-hot b
        result = sum(row[j] * E_b[j] for j in range(domain)) % mod

        # --- ALICE (knows keys, table, her plaintexts) ---

        # correction1: remove key_b contribution
        correction1 = sum(row[j] * K_b[j] for j in range(domain)) % mod

        # correction2: remove key_a noise that leaked through b's selection
        noise = [sum(T[i][j] * K_a[i] for i in range(domain)) % mod
                 for j in range(domain)]
        correction2 = sum(noise[j] * b_oh[j] for j in range(domain)) % mod

        decrypted = (result - correction1 - correction2) % mod
        expected = a * b
        status = "✓" if decrypted == expected else "✗"
        print(f"  {a} × {b} = {decrypted:2d}  (expected {expected:2d}) {status}")

  0 × 0 =  0  (expected  0) ✓
  0 × 1 =  0  (expected  0) ✓
  0 × 2 =  0  (expected  0) ✓
  0 × 3 =  0  (expected  0) ✓
  1 × 0 =  0  (expected  0) ✓
  1 × 1 =  1  (expected  1) ✓
  1 × 2 =  2  (expected  2) ✓
  1 × 3 =  3  (expected  3) ✓
  2 × 0 =  0  (expected  0) ✓
  2 × 1 =  2  (expected  2) ✓
  2 × 2 =  4  (expected  4) ✓
  2 × 3 =  6  (expected  6) ✓
  3 × 0 =  0  (expected  0) ✓
  3 × 1 =  3  (expected  3) ✓
  3 × 2 =  6  (expected  6) ✓
  3 × 3 =  9  (expected  9) ✓
